# Import


In [67]:
!bash pip install gym[box2d]

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [68]:
import torch
import gym
import numpy as np
from collections import deque
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# Utils

In [69]:
def from_tuple_to_tensor(tuple_of_np):
        tensor = np.zeros((len(tuple_of_np), tuple_of_np[0].shape[0], tuple_of_np[0].shape[1], tuple_of_np[0].shape[2]))
        for i, x in enumerate(tuple_of_np):
            tensor[i] = np.asarray(x)
        return tensor


class SumTree:
    def __init__(self, size):
        self.nodes = [0] * (2 * size - 1)
        self.data = [None] * size

        self.size = size
        self.count = 0
        self.real_size = 0

    @property
    def total(self):
        return self.nodes[0]

    def update(self, data_idx, value):
        idx = data_idx + self.size - 1  # child index in tree array
        change = value - self.nodes[idx]

        self.nodes[idx] = value

        parent = (idx - 1) // 2
        while parent >= 0:
            self.nodes[parent] += change
            parent = (parent - 1) // 2

    def add(self, value, data):
        self.data[self.count] = data

        self.update(self.count, value)    

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def get(self, cumsum):
        assert cumsum <= self.total

        idx = 0
        while 2 * idx + 1 < len(self.nodes):
            left, right = 2*idx + 1, 2*idx + 2

            if cumsum <= self.nodes[left]:
                idx = left
            else:
                idx = right
                cumsum = cumsum - self.nodes[left]

        data_idx = idx - self.size + 1

        return data_idx, self.nodes[idx], self.data[data_idx]

    def get_size(self):
        print(f"Size: {self.size}")
        print(f"Real Size: {self.real_size}")
        return self.real_size

    def __repr__(self):
        return f"SumTree(nodes={self.nodes.__repr__()}, data={self.data.__repr__()})"


# Buffer

Prioritized_experience_replay_buffer

In [70]:

class Prioritized_experience_replay_buffer:

    def __init__(self,env, memory_size=50000, burn_in=10000,eps=1e-2, alpha=0, beta=0.1):
        self.env=env
        self.tree = SumTree(size=memory_size)
        self.memory_size = memory_size
        self.burn_in = burn_in
        self.count = 0
        self.real_size = 0
        self.replay_memory = deque(maxlen=memory_size)
        # PER params
        self.eps = eps  # minimal priority, prevents zero probabilities
        self.alpha = alpha  # determines how much prioritization is used, α = 0 corresponding to the uniform case
        self.beta = beta  # determines the amount of importance-sampling correction, b = 1 fully compensate for the non-uniform probabilities
        self.max_priority = eps  # priority for new samples, init as eps

        # transition: state, action, reward, next_state, done
        self.state = torch.empty(memory_size, env.observation_space._shape[0],env.observation_space._shape[1],env.observation_space._shape[2], dtype=torch.float)
        self.action = torch.empty(memory_size, env.action_space.n, dtype=torch.float)
        self.reward = torch.empty(memory_size, dtype=torch.float)
        self.next_state = torch.empty(memory_size, env.observation_space._shape[0],env.observation_space._shape[1],env.observation_space._shape[2], dtype=torch.float)
        self.done = torch.empty(memory_size, dtype=torch.int)

    def sample(self, batch_size=32):
        assert self.real_size >= batch_size, "buffer contains less samples than batch size"
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        sample_idxs, tree_idxs = [], []
        priorities = torch.empty(batch_size, 1, dtype=torch.float)

        # To sample a minibatch of size k, the range [0, p_total] is divided equally into k ranges.
        # Next, a value is uniformly sampled from each range. Finally the transitions that correspond
        # to each of these sampled values are retrieved from the tree. (Appendix B.2.1, Proportional prioritization)
        segment = self.tree.total / batch_size
        for i in range(batch_size):
            a, b = segment * i, segment * (i + 1)

            cumsum = random.uniform(a, b)
            # sample_idx is a sample index in buffer, needed further to sample actual transitions
            # tree_idx is a index of a sample in the tree, needed further to update priorities
            tree_idx, priority, sample_idx = self.tree.get(cumsum)

            priorities[i] = priority
            tree_idxs.append(tree_idx)
            sample_idxs.append(sample_idx)

        # Concretely, we define the probability of sampling transition i as P(i) = p_i^α / \sum_{k} p_k^α
        # where p_i > 0 is the priority of transition i. (Section 3.3)
        probs = priorities / self.tree.total

        # The estimation of the expected value with stochastic updates relies on those updates corresponding
        # to the same distribution as its expectation. Prioritized replay introduces bias because it changes this
        # distribution in an uncontrolled fashion, and therefore changes the solution that the estimates will
        # converge to (even if the policy and state distribution are fixed). We can correct this bias by using
        # importance-sampling (IS) weights w_i = (1/N * 1/P(i))^β that fully compensates for the non-uniform
        # probabilities P(i) if β = 1. These weights can be folded into the Q-learning update by using w_i * δ_i
        # instead of δ_i (this is thus weighted IS, not ordinary IS, see e.g. Mahmood et al., 2014).
        # For stability reasons, we always normalize weights by 1/maxi wi so that they only scale the
        # update downwards (Section 3.4, first paragraph)
        weights = (self.real_size * probs) ** -self.beta

        # As mentioned in Section 3.4, whenever importance sampling is used, all weights w_i were scaled
        # so that max_i w_i = 1. We found that this worked better in practice as it kept all weights
        # within a reasonable range, avoiding the possibility of extremely large updates. (Appendix B.2.1, Proportional prioritization)
        weights = weights / weights.max()

        batch = (
            self.state[sample_idxs].to(device),
            self.action[sample_idxs].to(device),
            self.reward[sample_idxs].to(device),
            self.next_state[sample_idxs].to(device),
            self.done[sample_idxs].to(device)
        )
        return batch, weights, tree_idxs

    def update_priorities(self, data_idxs, priorities):
        if isinstance(priorities, torch.Tensor):
            priorities = priorities.detach().cpu().numpy()

        for data_idx, priority in zip(data_idxs, priorities):
            # The first variant we consider is the direct, proportional prioritization where p_i = |δ_i| + eps,
            # where eps is a small positive constant that prevents the edge-case of transitions not being
            # revisited once their error is zero. (Section 3.3)
            priority = (priority + self.eps) ** self.alpha

            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

    def burn_in_capacity(self):
        return len(self.replay_memory) / self.burn_in

    def capacity(self):
        return len(self.replay_memory) / self.memory_size

    def add(self, state, action, reward,done, next_state):
        #state, action, reward, next_state, done = transition

        # store transition index with maximum priority in sum tree
        self.tree.add(self.max_priority, self.count)

        # store transition in the buffer
        self.state[self.count] = torch.as_tensor(state)
        self.action[self.count] = torch.as_tensor(action)
        self.reward[self.count] = torch.as_tensor(reward)
        self.next_state[self.count] = torch.as_tensor(next_state)
        self.done[self.count] = torch.as_tensor(done)

        # update counters
        self.count = (self.count + 1) % self.memory_size
        self.real_size = min(self.memory_size, self.real_size + 1)





# Neural Networks

In [75]:
class Net(nn.Module):
    def __init__(self, n_frames,n_actions, hidden_size=32, bias=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_frames = n_frames
        self.conv1 = nn.Conv2d(n_frames, hidden_size, 7)
        self.conv2 = nn.Conv2d(hidden_size, hidden_size, 5)
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, n_actions)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        # Global Max Pooling
        x = x.reshape(self.hidden_size, -1).max(axis=1).values

        # Layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=0)
        return x

class Q_network(nn.Module):

    def __init__(self, env,  learning_rate=1e-4):
        super(Q_network, self).__init__()

        n_outputs = env.action_space.n

        #self.network = Net( ?? , ??)
        print( env.observation_space._shape[0], env.action_space.n)
        self.network = Net( 1, env.action_space.n)
        print("Q network:")
        print(self.network)

        self.optimizer = torch.optim.Adam(self.network.parameters(),
                                          lr=learning_rate)

        self.gs = transforms.Grayscale()
        self.rs = transforms.Resize((64,64))

    def greedy_action(self, state):
        # greedy action = ??
        # greedy_a = 0
        qvals = self.get_qvals(state)
        greedy_a = torch.max(qvals, dim=-1)[1].item()
        return greedy_a

    def get_qvals(self, state):
        state=self.preproc_state(state)
        out = self.network(state)
        return out

    def preproc_state(self, state):
        # State Preprocessing
        state = state[:83,:].transpose(2,0,1) #Torch wants images in format (channels, height, width)
        state = torch.from_numpy(state).float()
        state = self.gs(state) # grayscale
        state = self.rs(state) # resize
        return state/255 # normalize   
        


# Policy

In [79]:
from torch.nn.modules import loss
import gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from copy import deepcopy
from collections import deque
from random import randrange
from torchvision import transforms
from os.path import exists

class Policy(nn.Module):
    continuous = False # you can change this

    def __init__(self, device=torch.device('cpu')):
        super(Policy, self).__init__()
        self.device = device
        self.env = gym.make('CarRacing-v2', continuous=self.continuous,render_mode="rgb_array")
        learning_rate = 0.001
        self.epsilon = 0.5
        self.batch_size = 64
        self.network = Q_network(self.env, learning_rate)
        self.target_network = deepcopy(self.network)
        self.buffer = Prioritized_experience_replay_buffer(self.env)
        self.window = 50
        self.reward_threshold = 800
        self.initialize()
        self.step_count = 0
        self.episode = 0


    def forward(self, x):
        # TODO
        return x
    
    def act(self, state):
        action = self.network.greedy_action(state)
        return action

    def train(self):
        self.gamma = 0.99
        #max_episodes = 10000
        max_episodes = 3
        network_update_frequency = 10
        network_sync_frequency = 200
        self.loss_function = nn.MSELoss()
        self.s_0= self.env.reset()

      

        # Populate replay buffer
        for element in range(self.batch_size):
            self.take_step(mode='explore')
        ep = 0
        training = True
        self.populate = False

        #load pretrained model
        if(exists('model.pth')):
          
          checkpoint = torch.load('model.pth')
          self.network.load_state_dict(checkpoint['model_state_dict'])
          self.network.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
          self.episode = checkpoint['epoch']
          self.update_loss = checkpoint['loss']
          self.epsilon = checkpoint['epsilon']

        while training:
            self.s_0 = self.env.reset()

            self.rewards = 0
            done = False
            while not done:
                if ((ep % 5) == 0):
                    self.env.render()

                p = np.random.random()
                if p < self.epsilon:
                    done = self.take_step(mode='explore')
                    # print("explore")
                else:
                    done = self.take_step(mode='exploit')
                    # print("train")
                # Update network
                if self.step_count % network_update_frequency == 0:
                    self.update()
                # Sync networks
                if self.step_count % network_sync_frequency == 0:
                    self.target_network.load_state_dict(
                        self.network.state_dict())
                    self.sync_eps.append(ep)

                if done:
                    if self.epsilon >= 0.05:
                        self.epsilon = self.epsilon * 0.7
                    ep += 1
                    if self.rewards > 2000:
                        self.training_rewards.append(2000)
                    elif self.rewards > 1000:
                        self.training_rewards.append(1000)
                    elif self.rewards > 500:
                        self.training_rewards.append(500)
                    else:
                        self.training_rewards.append(self.rewards)
                    if len(self.update_loss) == 0:
                        self.training_loss.append(0)
                    else:
                        self.training_loss.append(np.mean(self.update_loss))
                    self.update_loss = []
                    mean_rewards = np.mean(self.training_rewards[-self.window:])
                    mean_loss = np.mean(self.training_loss[-self.window:])
                    self.mean_training_rewards.append(mean_rewards)
                    print(
                        "\rEpisode {:d} Mean Rewards {:.2f}  Episode reward = {:.2f}   mean loss = {:.2f}\t\t".format(
                            ep, mean_rewards, self.rewards, mean_loss), end="")

                    if ep >= max_episodes:
                        training = False
                        print('\nEpisode limit reached.')
                        break
                    if mean_rewards >= self.reward_threshold:
                        training = False
                        print('\nEnvironment solved in {} episodes!'.format(
                            ep))
                        # break

                    #save trainable model    
                    
                    torch.save({'epoch': ep,'model_state_dict': self.network.state_dict(),
                    'optimizer_state_dict': self.network.optimizer.state_dict(),
                    'loss': self.update_loss, 'epsilon': self.epsilon}, 'model.pth')

                    #save non trainable model
                    self.save()
        # save models
        self.save()
        return




    def take_step(self, mode='exploit'):
        # choose action with epsilon greedy
        if mode == 'explore':
                action = self.env.action_space.sample()
        else:
                action = self.network.greedy_action(self.s_0)

        #simulate action
        s_1, r, done, _ = self.env.step(action)


        #put experience in the buffer
        self.buffer.add(self.s_0, action, r, done, s_1)

        self.rewards += r

        self.s_0 = s_1.copy()

        self.step_count += 1
        if done:

            self.s_0= self.env.reset()
        return done


    def update(self):
        self.network.optimizer.zero_grad()
        batch,weights,tree_idxs = self.buffer.sample(batch_size=self.batch_size)
        loss,td_error = self.calculate_loss(batch,weights=weights)

        loss.backward()
        self.network.optimizer.step()

        self.update_loss.append(loss.item())

    def initialize(self):
        self.training_rewards = []
        self.training_loss = []
        self.update_loss = []
        self.mean_training_rewards = []
        self.sync_eps = []
        self.rewards = 0
        self.step_count = 0


    def calculate_loss(self, batch,weights=None):
        #extract info from batch
        states, actions, rewards, next_states, dones = list(batch)
        #transform in torch tensors
        rewards = torch.FloatTensor(rewards).reshape(-1, 1)
        actions = torch.LongTensor(np.array(actions[:,-1],dtype="int64")).reshape(-1, 1)
        dones = torch.IntTensor(dones).reshape(-1, 1)
        states = from_tuple_to_tensor(states)
        next_states = from_tuple_to_tensor(next_states)



        ###############
        # DDQN Update #
        ###############
        # Q(s,a) = ??
        loss=0
        for i,s in enumerate(states):
            qvals = self.network.get_qvals(s)
            #TODO Why 0 in line below instead of 1
            qvals = torch.gather(qvals, 0, actions[i])
            next_qvals= self.target_network.get_qvals(next_states[i])
            next_qvals_max = torch.max(next_qvals, dim=-1)[0].reshape(-1, 1)
            target_qvals = rewards[i] + (1 - dones[i])*self.gamma*next_qvals_max
            #TODO update loss with PER
            loss+=torch.mean((qvals - target_qvals) ** 2 * weights)

        loss=loss/len(states)

        if weights is None:
            weights = torch.ones_like(qvals)

        td_error = torch.abs(qvals - target_qvals).detach()
        #loss = torch.mean((qvals - target_qvals) ** 2 * weights)


        # loss = self.loss_function( Q(s,a) , target_Q(s,a))
        #TODO loss = self.loss_function(qvals, target_qvals)

        return loss,td_error

    def save(self):
        torch.save(self.state_dict(), 'model.pt')

    def load(self):
        self.load_state_dict(torch.load('model.pt'))

    def to(self, device):
        ret = super().to(device)
        ret.device = device
        return ret


Main

In [80]:
import argparse
import random
import numpy as np
import gym


def evaluate(env=None, n_episodes=1, render=True):
    agent = Policy()
    agent.load()

    max_steps_per_episode = 1000

    env = gym.make('CarRacing-v2', continuous=agent.continuous)
    if render:
        env = gym.make('CarRacing-v2', continuous=agent.continuous, render_mode='human')
        
    rewards = []
    for episode in range(n_episodes):
        total_reward = 0
        done = False
        s = env.reset()
        for i in range(max_steps_per_episode):
            action = agent.act(s)
            
            s, reward, done, truncated, info = env.step(action)
            if render: env.render()
            total_reward += reward
            if done or truncated: break
        
        rewards.append(total_reward)
        
    print('Mean Reward:', np.mean(rewards))

def train():
    agent = Policy()
    agent.train()
    agent.save()




# Train

In [ ]:
train()

96 5
Q network:
Net(
  (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=32, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=5, bias=True)
)
Episode 1 Mean Rewards -83.82  Episode reward = -83.82   mean loss = 0.32		

In [ ]:
evaluate(render=True)